# 10. 양성 대조: 파형 모양 피처에서 약지도 모델이 신호를 잡는가
1장에서 위치를 통제하고도 남았던 신호(동저항 하강폭, 전류 후반/초반비, 가압력 최대 등)는 파형 모양에서 나온다. KAMP에는 없는 정보다.
목적 1: 평가 코드가 신호가 있을 때 제대로 잡는지(양성 대조). 목적 2: 날짜별 개수만으로 학습하는 우리 모델이 신호가 있을 때 그걸 잡는지.
입력은 파형 모양 피처 10개를 위치 평균 제거(`shape_locstd`). 기준은 02와 같다.

In [1]:
import sys, time, numpy as np, pandas as pd, torch
sys.path.insert(0, ".")
from joblib import Parallel, delayed
from wl import runner, bags
pd.set_option("display.width", 250)
MODELS = ["const_rate", "autoencoder", "A_dllp_mlp", "B_count_mlp", "C_prior_two", "D_pseudo_logreg", "UB_logreg", "UB_catboost"]
CONDS = list(bags.CONDITIONS); SEEDS = list(range(5)); N_GPU = torch.cuda.device_count()
name = "shape_control"
def _task(i, m, c, s): return runner.run_one(m, c, s, feature_set="shape_locstd", gpu=i % max(N_GPU, 1))
if (runner.RES / f"{name}.pkl").exists():
    runs = runner.load_runs(name)
else:
    tasks = [(m, c, s) for m in MODELS for c in CONDS for s in SEEDS]
    t = time.time(); runs = Parallel(n_jobs=8, verbose=2)(delayed(_task)(i, *k) for i, k in enumerate(tasks))
    print(f"{len(runs)}개 실행 {time.time()-t:.0f}초"); runner.save_runs(runs, name)
by = {}
for r in runs: by.setdefault((r["model"], r["cond"]), []).append(r)
boots = dict(Parallel(n_jobs=16)(delayed(lambda k: (k, runner.bootstrap(by[k], n_boot=200)))(k) for k in by))
def fmt(m, c, k):
    r = boots[(m, c)].loc[k]; return f"{r['mean']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]"
KEYS = ["auroc", "wl_auroc", "recall@0.1", "wl_recall@0.1", "recall@5x"]
for c in ("orig", "k8_1"):
    print(f"\n=== shape_locstd | {c} ==="); display(pd.DataFrame({k: {m: fmt(m, c, k) for m in MODELS} for k in KEYS}))

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    7.2s


C:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[Parallel(n_jobs=8)]: Done 160 out of 160 | elapsed:  3.5min finished


160개 실행 209초



=== shape_locstd | orig ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.196 [0.192, 0.202]"
autoencoder,"0.504 [0.452, 0.560]","0.533 [0.446, 0.629]","0.099 [0.046, 0.143]","0.111 [0.077, 0.161]","0.171 [0.112, 0.238]"
A_dllp_mlp,"0.638 [0.575, 0.697]","0.638 [0.562, 0.698]","0.339 [0.209, 0.421]","0.216 [0.149, 0.242]","0.511 [0.427, 0.599]"
B_count_mlp,"0.648 [0.603, 0.690]","0.645 [0.574, 0.704]","0.273 [0.191, 0.347]","0.181 [0.126, 0.219]","0.445 [0.363, 0.516]"
C_prior_two,"0.677 [0.612, 0.731]","0.665 [0.569, 0.739]","0.307 [0.160, 0.391]","0.208 [0.136, 0.251]","0.505 [0.397, 0.575]"
D_pseudo_logreg,"0.679 [0.607, 0.728]","0.677 [0.579, 0.745]","0.289 [0.182, 0.371]","0.198 [0.140, 0.245]","0.433 [0.291, 0.520]"
UB_logreg,"0.671 [0.617, 0.723]","0.671 [0.579, 0.734]","0.314 [0.180, 0.400]","0.203 [0.130, 0.242]","0.434 [0.288, 0.514]"
UB_catboost,"0.743 [0.699, 0.798]","0.636 [0.557, 0.726]","0.430 [0.358, 0.530]","0.153 [0.096, 0.205]","0.520 [0.435, 0.660]"



=== shape_locstd | k8_1 ===


,auroc,wl_auroc,recall@0.1,wl_recall@0.1,recall@5x
const_rate,"0.500 [0.500, 0.500]","0.500 [0.500, 0.500]","0.100 [0.100, 0.100]","0.100 [0.100, 0.100]","0.049 [0.047, 0.050]"
autoencoder,"0.483 [0.407, 0.557]","0.516 [0.409, 0.643]","0.084 [0.033, 0.147]","0.128 [0.065, 0.220]","0.025 [0.000, 0.077]"
A_dllp_mlp,"0.605 [0.548, 0.667]","0.621 [0.534, 0.720]","0.147 [0.055, 0.241]","0.171 [0.099, 0.254]","0.095 [0.018, 0.172]"
B_count_mlp,"0.526 [0.457, 0.601]","0.506 [0.383, 0.628]","0.126 [0.071, 0.188]","0.103 [0.058, 0.164]","0.084 [0.009, 0.145]"
C_prior_two,"0.519 [0.463, 0.594]","0.519 [0.437, 0.633]","0.137 [0.082, 0.217]","0.122 [0.079, 0.175]","0.137 [0.029, 0.216]"
D_pseudo_logreg,"0.587 [0.509, 0.651]","0.613 [0.504, 0.722]","0.095 [0.028, 0.230]","0.180 [0.110, 0.249]","0.032 [0.000, 0.102]"
UB_logreg,"0.632 [0.552, 0.690]","0.617 [0.527, 0.723]","0.242 [0.117, 0.348]","0.173 [0.098, 0.243]","0.184 [0.065, 0.288]"
UB_catboost,"0.663 [0.604, 0.714]","0.473 [0.372, 0.595]","0.347 [0.264, 0.429]","0.112 [0.058, 0.180]","0.208 [0.118, 0.277]"


In [2]:
rows = []
for m in MODELS[1:]:
    row = {"모델": m}
    for c in CONDS:
        b = boots[(m, c)]
        row[c] = ("O" if b.loc["recall@0.1", "lo"] > .10 else "X") + ("O" if (b.loc["wl_recall@0.1", "lo"] > .10 and b.loc["wl_auroc", "lo"] > .5) else "X")
    rows.append(row)
print("[기준2 포착률][기준3 같은 위치 안]"); pd.DataFrame(rows)

[기준2 포착률][기준3 같은 위치 안]


,모델,orig,k8_4,k8_1,k8_05
0,autoencoder,XX,XX,XX,XX
1,A_dllp_mlp,OO,OO,XX,XX
2,B_count_mlp,OO,XX,XX,XX
3,C_prior_two,OO,OX,XX,XX
4,D_pseudo_logreg,OO,OO,XO,XX
5,UB_logreg,OO,OO,OX,OO
6,UB_catboost,OX,OX,OX,OX
